## IDP On Resume Dataset

### Assignment - Consider 5 resume from kaglle resume dataset and convert them to text using IDP

### Resume Dataset loading using kaggle

In [ ]:
import kagglehub

path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")
print("Dataset downloaded to:", path)
!ls -lh "$path"

### Pick up 5 random resumes

In [ ]:
import os
import random

# Main resume folder
resume_dir = "/kaggle/input/resume-dataset/data/data"

# Collect all resume file paths from all subfolders
all_resumes = []
for category in os.listdir(resume_dir):
    category_path = os.path.join(resume_dir, category)
    if os.path.isdir(category_path):  # only look inside folders
        for f in os.listdir(category_path):
            file_path = os.path.join(category_path, f)
            all_resumes.append(file_path)

print("Total resumes found:", len(all_resumes))

# Pick random 5 resumes
sample_resumes = random.sample(all_resumes, 5)
print("Randomly chosen resumes:")
for resume in sample_resumes:
    print(resume)

### PDF to image

In [ ]:
! pip install pdf2image
!apt-get install -y poppler-utils

In [ ]:
from pdf2image import convert_from_path
import os

resume_photos = []
output_dir = "resume_images"
os.makedirs(output_dir, exist_ok=True)

for idx, resume in enumerate(sample_resumes):
    images = convert_from_path(resume)
    for i, page in enumerate(images):
        filename = f"resume_{idx}_page_{i}.png"
        filepath = os.path.join(output_dir, filename)
        page.save(filepath, "PNG")
        resume_photos.append(filepath)

print("Converted images:", resume_photos)


### Image before Pre-processing

In [ ]:
display(images[0])

In [ ]:
from IPython.display import display

# Suppose 'images' is the list of PIL.Image objects
for i, page in enumerate(images):
    print(f"Page {i+1}")
    display(page)

### Image Processing
Gray Scale Conversion, Noise reduction, binarization, Deskewing

In [ ]:
import cv2
import numpy as np
from PIL import Image
from pdf2image import convert_from_path
import os
import time

In [ ]:
# Output folder
output_dir = "final_processed_resumes"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
def convert_to_grayscale(pil_image):
    image_array = np.array(pil_image)
    gray = cv2.cvtColor(image_array, cv2.COLOR_RGB2GRAY)
    return gray

In [ ]:
def apply_gaussian_blur(gray_image, kernel_size=(5,5)):
    return cv2.GaussianBlur(gray_image, kernel_size, 0)

In [ ]:
def binarize_image(blur_image):
    return cv2.adaptiveThreshold(
        blur_image,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,  # Invert colors
        11,  # block size
        4    # constant C
    )

In [ ]:
import cv2
import numpy as np
import math

def deskew_image_final_bypass(binary_image, skew_threshold=1.0):
    """
    Corrects only minor skew. Crucially, it bypasses ALL rotation if the image
    is already near upright (0 or 90 degrees), based on the raw minAreaRect angle.
    """
    coords = cv2.findNonZero(binary_image)
    if coords is None:
        return binary_image, 0.0

    rect = cv2.minAreaRect(coords)
    (x, y), (w, h), raw_angle = rect

    # --- 1. BYPASS CHECK FOR ALREADY UPRIGHT IMAGES ---

    # If the text is detected as nearly vertical (your 85-90 degrees) OR
    # nearly horizontal (0-5 degrees), we consider it UPGRIGHT and bypass deskewing.

    # Normalize the raw angle to be between 0 and 90 degrees absolute for easy checking
    abs_angle = abs(raw_angle)

    # Check 1: Image is near vertical (e.g., 85 to 95 degrees)
    is_near_vertical = abs(abs_angle - 90.0) < 5.0

    # Check 2: Image is near horizontal (e.g., 0 to 5 degrees)
    is_near_horizontal = abs_angle < 5.0

    # For a portrait image that's already upright, it will pass the is_near_vertical check.
    if is_near_vertical or is_near_horizontal:
        # If the image is already upright, force the rotation angle to 0 and return the original image.
        return binary_image, 0.0

    # --- 2. CALCULATE MINOR SKEW CORRECTION (Only runs if the image is clearly skewed, e.g., 45 degrees) ---

    # Normalizing the angle to the standard [-45, 45] range.

    # Ensure angle is in the standard [-90, 0) range for correction logic (handle your non-standard 90)
    if raw_angle > 0:
        normalized_angle = raw_angle - 180.0
    else:
        normalized_angle = raw_angle

    # Apply standard deskew logic to get the final small rotation amount
    if normalized_angle < -45:
        rotation_angle = -(90 + normalized_angle)
    else:
        rotation_angle = -normalized_angle

    # If the remaining minor rotation is less than the threshold, also bypass
    if abs(rotation_angle) < skew_threshold:
        return binary_image, 0.0

    # --- 3. ROTATE THE IMAGE (Only for genuinely skewed images) ---

    (img_h, img_w) = binary_image.shape[:2]
    center = (img_w // 2, img_h // 2)
    M = cv2.getRotationMatrix2D(center, rotation_angle, 1.0)

    # Calculate new dimensions
    cos = np.abs(M[0, 0])
    sin = np.abs(M[0, 1])
    nW = int((img_h * sin) + (img_w * cos))
    nH = int((img_h * cos) + (img_w * sin))
    M[0, 2] += (nW / 2) - center[0]
    M[1, 2] += (nH / 2) - center[1]

    deskewed = cv2.warpAffine(binary_image, M, (nW, nH),
                              flags=cv2.INTER_CUBIC,
                              borderMode=cv2.BORDER_REPLICATE)

    return deskewed, rotation_angle

In [ ]:
for r_idx, resume in enumerate(sample_resumes):
    start_resume = time.time()
    images = convert_from_path(resume)
    print(f"\nProcessing Resume {r_idx+1}/{len(sample_resumes)}: {resume}, Pages: {len(images)}")

    for p_idx, page in enumerate(images):
        step_times = {}

        # Grayscale
        t0 = time.time()
        gray = convert_to_grayscale(page)
        step_times['grayscale'] = time.time() - t0

        # Gaussian blur
        t1 = time.time()
        blurred = apply_gaussian_blur(gray)
        step_times['gaussian_blur'] = time.time() - t1

        # Binarization
        t2 = time.time()
        binary = binarize_image(blurred)
        step_times['binarization'] = time.time() - t2

        # Deskew using new bypass function
        t3 = time.time()
        deskewed, angle = deskew_image_final_bypass(binary, skew_threshold=1.0)
        step_times['deskew'] = time.time() - t3

        # Save processed image
        filename = f"resume_{r_idx}_page_{p_idx}_processed.png"
        filepath = os.path.join(output_dir, filename)
        Image.fromarray(deskewed).save(filepath)

        print(f"  Page {p_idx+1}: Skew={angle:.2f}° | Step times (s): {step_times}")

    print(f"Finished Resume {r_idx+1} in {time.time() - start_resume:.2f} seconds")


### Tesseract

In [ ]:
from PIL import Image
import pytesseract
import os
import time

# Folder where processed resume images are saved
input_folder_path = "final_processed_resumes"

# Folder to save extracted text
output_folder_path = "tesseract_output"
os.makedirs(output_folder_path, exist_ok=True)
print(f"Text output folder: {output_folder_path}")

# List all processed images
processed_images = [f for f in os.listdir(input_folder_path) if f.lower().endswith((".png"))]
total_images = len(processed_images)
print(f"Total processed images: {total_images}")

start_time = time.time()

# Loop through images and extract text
for i, image_name in enumerate(processed_images, 1):
    print(f"Processing image {i}/{total_images}: {image_name}")
    image_path = os.path.join(input_folder_path, image_name)

    # Extract text using pytesseract
    text = pytesseract.image_to_string(Image.open(image_path))

    # Save extracted text
    txt_filename = image_name.rsplit(".", 1)[0] + ".txt"
    output_path = os.path.join(output_folder_path, txt_filename)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(text)

    print(f"Saved extracted text to: {output_path}")
    print("-" * 50)

print("Text extraction completed!")
print(f"Total time taken: {time.time() - start_time:.2f} seconds")


### Information Extraction

In [ ]:
prompt = """
###ROLE###
You are a world-class Resume Information Extraction Specialist. You are an expert at analyzing resumes in various formats (PDFs, images, scanned documents) and extracting structured, accurate, and relevant candidate information for recruitment purposes.

###CONTEXT###
The user has a set of resume images that have already been pre-processed using OpenCV: converted to grayscale, noise-reduced, binarized, and deskewed. Text has been extracted from these images using Tesseract OCR, which may contain minor OCR errors. You should use this extracted text as supporting evidence to extract accurate information from the resumes.

###TASK###
1. Analyze the Tesseract-extracted text carefully.
2. Extract the following fields:
   - `name`: Full name of the candidate
   - `email`: Candidate's email address
   - `phone`: Contact number
   - `education`: Degrees, institutions, and graduation years
   - `skills`: List of technical and non-technical skills
   - `experience`: Previous job roles, companies, and durations
   - `projects`: Relevant projects with brief descriptions (if any)
3. Correct any OCR errors as needed to ensure accurate extraction (e.g., misread characters in emails, phone numbers, or names).
4. If a field is missing or cannot be confidently determined, leave it as an empty string or empty list.
5. Focus on extracting content only; ignore formatting, layout, or minor skew.

###CONSTRAINTS###
- Do not include any extra commentary, explanations, or filler text.
- Do not return raw OCR text; only provide the structured, cleaned information.
- Avoid guessing information not present in the resume.
- Ensure consistency in formatting (e.g., phone numbers, emails).

###OUTPUT FORMAT###
Provide the output strictly as a JSON object in this format:
{
    "name": "FULL_NAME",
    "email": "EMAIL_ADDRESS",
    "phone": "PHONE_NUMBER",
    "education": ["Degree, Institution, Year", "..."],
    "skills": ["Skill1", "Skill2", "..."],
    "experience": ["Role, Company, Duration", "..."],
    "projects": ["Project Name: Brief Description", "..."]
}

###EXAMPLES###
Input Text (from Tesseract OCR):
"John Doe\nEmail: john.doe@example.com\nPhone: +1-234-567-8901\nEducation: B.Tech in CS, MIT, 2020\nSkills: Python, Machine Learning\nExperience: Software Engineer, ABC Corp, 2020-2023\nProjects: Resume Parser Project"

Output JSON:
{
    "name": "John Doe",
    "email": "john.doe@example.com",
    "phone": "+1-234-567-8901",
    "education": ["B.Tech in CS, MIT, 2020"],
    "skills": ["Python", "Machine Learning"],
    "experience": ["Software Engineer, ABC Corp, 2020-2023"],
    "projects": ["Resume Parser Project"]
}

Here is the OCR-extracted text (use this as support for extracting information):
"""

In [ ]:
from google import genai
from google.colab import userdata
import json

In [ ]:
genai_client = genai.Client(api_key=userdata.get('GOOGLE_GENAI_API'))

In [ ]:
import os
import time
import json
from PIL import Image

image_folder_path = "final_processed_resumes"
text_folder_path = "tesseract_output"
output_folder_path = "json_output"

os.makedirs(output_folder_path, exist_ok=True)
print(f"JSON output folder: {output_folder_path}")

# List all PNG images
processed_images = [f for f in os.listdir(image_folder_path) if f.lower().endswith(".png")]
total_images = len(processed_images)
print(f"Total images to process: {total_images}")

start_time = time.time()

for i, image_name in enumerate(processed_images, 1):
    print(f"\nProcessing image {i}/{total_images}: {image_name}")

    # Load image
    image_path = os.path.join(image_folder_path, image_name)
    print(f"Loading image: {image_path}")
    image = Image.open(image_path)

    # Load corresponding OCR text
    text_filename = image_name.replace(".png", ".txt")
    text_path = os.path.join(text_folder_path, text_filename)
    print(f"Loading OCR-extracted text: {text_path}")
    with open(text_path, "r", encoding="utf-8") as f:
        ocr_text = f.read()

    # Combine prompt template with OCR text
    current_prompt = prompt + "\n" + ocr_text

    # Prepare contents for Gemini model
    contents = [
        image,
        {"text": current_prompt}
    ]

    # Generate structured JSON
    response = genai_client.models.generate_content(model='gemini-flash-lite-latest', contents=contents)

    # Token usage info
    usage_metadata = response.usage_metadata
    print(f"Input Token Count: {usage_metadata.prompt_token_count}")
    print(f"Thoughts Token Count: {usage_metadata.thoughts_token_count}")
    print(f"Output Token Count: {usage_metadata.candidates_token_count}")
    print(f"Total Token Count: {usage_metadata.total_token_count}")

    # Parse and save JSON output
    try:
        extracted_information = json.loads(response.text.replace('```json', '').replace('```', ''))
    except json.JSONDecodeError:
        print("Warning: Failed to parse JSON, saving raw response.")
        extracted_information = {"raw_response": response.text}

    output_path = os.path.join(output_folder_path, image_name.replace(".png", ".json"))
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(extracted_information, f, indent=4)

    print(f"Saved extracted information to {output_path}")
    print("-" * 50)

    # Optional: delay to prevent rate limits
    time.sleep(60)

print("Information extraction completed!")
print(f"Total time taken: {time.time() - start_time:.2f} seconds")

### Extracted texts

In [ ]:
output_folder_path = "json_output"

# List all JSON files
json_files = [f for f in os.listdir(output_folder_path) if f.lower().endswith(".json")]
json_files.sort()

print(f"Total JSON files to print: {len(json_files)}\n")

for i, json_file in enumerate(json_files, 1):
    json_path = os.path.join(output_folder_path, json_file)

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"=== Resume {i}: {json_file} ===")
    # Pretty-print JSON with indentation
    print(json.dumps(data, indent=4, ensure_ascii=False))
    print("="*80)
